# 07 — LLM ingredient matcher (free open model, Google Colab)

**Goal:** measure the LLM matcher (`src/llm_matching.py`) on the same 50 TEST ingredients as RapidFuzz (7/50) and AI rev 2 (12/50).

**How it works:** the LLM translates the ingredient (EN/ES/CA) into English + French names → RapidFuzz retrieves candidate Agribalyse foods → the LLM chooses one code or NONE.

**Decided BEFORE running (do not change after seeing TEST):**
- Model: `Qwen/Qwen2.5-7B-Instruct`, 4-bit, greedy decoding (free, open, reproducible)
- Objective for the threshold: `accuracy` (same as log 07, so the numbers are comparable)
- The threshold is chosen on TUNE only; TEST is evaluated once.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*. It takes about 10–20 minutes.

In [ ]:
# 1. Check that a GPU is available (should show a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code (branch llm-matcher) and install the packages
!git clone -q -b llm-matcher https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes

In [ ]:
# 3. Run the evaluation (downloads the model the first time, ~5 GB)
!python src/evaluate_matching.py --matcher llm --provider hf --details

## Results

Copy the numbers printed above into the dev log:
- **TEST correct** (exact Agribalyse code) — compare with RapidFuzz 7/50 and AI rev 2 12/50
- **CO2e OK** (right food, or CO2e within ±25%) — compare with RapidFuzz 10/50 and AI rev 2 19/50
- **Confident errors** (wrong food WITH a CO2e number) — compare with AI rev 2: 25
- Results by language (EN / ES / CA)

In [ ]:
# 4. The remaining errors, to write the error analysis
import pandas as pd
r = pd.read_csv("data/results_llm_Qwen2.5-7B-Instruct_test.csv")
pd.set_option("display.width", 200, "display.max_colwidth", 45)
print("Confident errors (wrong food with a CO2e number):")
print(r[r.confident_error][["ingredient_text", "language", "agribalyse_name", "pred_name", "co2e_ok"]].to_string(index=False))
print("\nUncertain (no number shown):")
print(r[r.uncertain & ~r.correct][["ingredient_text", "language", "agribalyse_name"]].to_string(index=False))

In [ ]:
# 5. Download the evidence to add to the repo (data/ folder)
from google.colab import files
files.download("data/results_llm_Qwen2.5-7B-Instruct_test.csv")
files.download("data/llm_cache.json")